In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense, Flatten
from keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # ← fix
import matplotlib.pyplot as plt

In [3]:
# ── Paths ─────────────────────────────────────────────────────────────────────
TRAIN_DIR = '/content/drive/MyDrive/ML_DATASETS/cats_dogs_light/final_train'

In [6]:
# ── Build Model (VGG16 as frozen feature extractor) ───────────────────────────
conv_base = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(150, 150, 3)
)
conv_base.trainable = False   # freeze — we only train the top layers

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [7]:

model = Sequential([
    conv_base,
    Flatten(),
    Dense(256, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [8]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,812,353 (64.13 MB)

 Trainable params: 2,097,665 (8.00 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [9]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2        # 20% of final_train used as validation
)

train_generator = datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary',
    subset='training',          # 80%
    shuffle=True
)

validation_generator = datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary',
    subset='validation',        # 20%
    shuffle=False
)

Found 8002 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.


In [10]:
print("Classes:", train_generator.class_indices)   # {'cats': 0, 'dogs': 1}

Classes: {'cats': 0, 'dogs': 1}


In [11]:
# ── Compile & Train ───────────────────────────────────────────────────────────
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator
)


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 2654s 11s/step - accuracy: 0.7766 - loss: 0.5016 - val_accuracy: 0.7785 - val_loss: 0.4997
Epoch 2/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 39s 154ms/step - accuracy: 0.9021 - loss: 0.2339 - val_accuracy: 0.8390 - val_loss: 0.4096
Epoch 3/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 37s 147ms/step - accuracy: 0.9259 - loss: 0.1754 - val_accuracy: 0.8260 - val_loss: 0.4333
Epoch 4/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 39s 153ms/step - accuracy: 0.9542 - loss: 0.1116 - val_accuracy: 0.8600 - val_loss: 0.4123
Epoch 5/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 38s 151ms/step - accuracy: 0.9814 - loss: 0.0633 - val_accuracy: 0.8480 - val_loss: 0.4387
Epoch 6/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 38s 150ms/step - accuracy: 0.9836 - loss: 0.0583 - val_accuracy: 0.7905 - val_loss: 0.7133
Epoch 7/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 38s 150ms/step - accuracy: 0.9769 - loss: 0.0600 - val_accuracy: 0.8515 - val_loss: 0.5304
Epoch 8/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 38s 149ms/step - accuracy: 0.9937 - loss: 0

In [ ]:
# ── Plot Accuracy ─────────────────────────────────────────────────────────────
plt.plot(history.history['accuracy'],     color='red',  label='train')
plt.plot(history.history['val_accuracy'], color='blue', label='validation')
plt.title('Model Accuracy')
plt.legend()
plt.show()

# ── Plot Loss ─────────────────────────────────────────────────────────────────
plt.plot(history.history['loss'],     color='red',  label='train')
plt.plot(history.history['val_loss'], color='blue', label='validation')
plt.title('Model Loss')
plt.legend()
plt.show()